# Media Format Converter - Google Colab Version

This notebook allows you to convert MP4 files to MP3 format using Google Colab.

## Features:
- Single file conversion
- Batch conversion
- Automatic dependency installation
- File upload/download support

## Usage:
1. Run the setup cell to install dependencies
2. Upload your MP4 files
3. Run the conversion cell
4. Download the converted MP3 files

## 1. Setup - Install Dependencies

Run this cell first to install all required dependencies.

In [ ]:
import sys
import subprocess
import os

print("=" * 60)
print("Setting up Media Format Converter...")
print("=" * 60)
print()

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running in local environment")

print()

# Install ffmpeg
print("Installing ffmpeg...")
result = subprocess.run(['apt-get', 'update'], capture_output=True)
result = subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)

# Verify ffmpeg installation
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ ffmpeg installed successfully")
    # Extract version
    version_line = result.stdout.split('\n')[0]
    print(f"  {version_line}")
else:
    print("✗ ffmpeg installation failed")
    sys.exit(1)

print()

# Install Python packages
print("Installing Python packages...")
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ffmpeg-python'], 
                       capture_output=True)
if result.returncode == 0:
    print("✓ ffmpeg-python installed successfully")
else:
    print("✗ ffmpeg-python installation failed")
    sys.exit(1)

print()
print("=" * 60)
print("Setup completed successfully!")
print("=" * 60)

## 2. Converter Functions

Run this cell to load the converter functions.

In [ ]:
import os
import ffmpeg
import glob
from pathlib import Path
from IPython.display import display, HTML

def validate_input_file(file_path):
    """Validate if the input file exists and has correct extension"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    if not file_path.lower().endswith('.mp4'):
        raise ValueError("Input file must be MP4 format")
    
    return True


def convert_mp4_to_mp3(input_file, output_file=None, overwrite=True):
    """
    Convert MP4 file to MP3 format
    
    Args:
        input_file (str): Path to input MP4 file
        output_file (str): Path to output MP3 file (optional)
        overwrite (bool): Overwrite if output file exists
    
    Returns:
        str: Path to the output MP3 file, or None if failed
    """
    # Validate input file
    validate_input_file(input_file)
    
    # Generate output filename if not provided
    if output_file is None:
        input_path = Path(input_file)
        output_file = str(input_path.with_suffix('.mp3'))
    else:
        # Ensure output file has .mp3 extension
        if not output_file.lower().endswith('.mp3'):
            output_file += '.mp3'
    
    # Check if output file already exists
    if os.path.exists(output_file) and not overwrite:
        print(f"⊘ Skipped (file exists): {output_file}")
        return None
    
    try:
        print(f"Converting: {input_file} -> {output_file}")
        
        # Convert using ffmpeg
        stream = ffmpeg.input(input_file)
        stream = ffmpeg.output(stream, output_file, acodec='libmp3lame', audio_bitrate='192k')
        ffmpeg.run(stream, overwrite_output=True, quiet=True)
        
        print(f"✓ Conversion completed: {output_file}")
        return output_file
    
    except ffmpeg.Error as e:
        print(f"✗ Conversion failed: {e.stderr.decode() if e.stderr else str(e)}")
        return None
    except Exception as e:
        print(f"✗ Error occurred: {str(e)}")
        return None


def batch_convert(input_pattern, output_dir=None, overwrite=True):
    """
    Batch convert multiple MP4 files to MP3 format
    
    Args:
        input_pattern (str): File pattern or directory path
        output_dir (str): Output directory for converted files (optional)
        overwrite (bool): Overwrite existing files
    
    Returns:
        tuple: (success_count, fail_count, skip_count)
    """
    # Find all MP4 files matching the pattern
    if os.path.isdir(input_pattern):
        search_pattern = os.path.join(input_pattern, "*.mp4")
        files = glob.glob(search_pattern)
    else:
        files = glob.glob(input_pattern)
    
    if not files:
        print(f"No files found matching: {input_pattern}")
        return 0, 0, 0
    
    print(f"Found {len(files)} file(s)")
    print()
    
    success_count = 0
    fail_count = 0
    skip_count = 0
    
    for i, input_file in enumerate(files, 1):
        print(f"[{i}/{len(files)}] Processing: {input_file}")
        
        # Determine output file path
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            output_file = os.path.join(output_dir, Path(input_file).stem + '.mp3')
        else:
            output_file = None
        
        try:
            result = convert_mp4_to_mp3(input_file, output_file, overwrite=overwrite)
            if result:
                success_count += 1
            else:
                skip_count += 1
        except Exception as e:
            fail_count += 1
            print(f"✗ Processing failed: {input_file}")
            continue
        
        print()
    
    return success_count, fail_count, skip_count


print("✓ Converter functions loaded successfully")

## 3. Upload MP4 Files

Run this cell to upload your MP4 files to Colab.

In [ ]:
from google.colab import files
import os

print("=" * 60)
print("Upload MP4 Files")
print("=" * 60)
print()
print("Please select one or more MP4 files to upload...")
print()

uploaded = files.upload()

print()
print(f"✓ Uploaded {len(uploaded)} file(s)")
for filename in uploaded.keys():
    print(f"  - {filename} ({len(uploaded[filename])} bytes)")

## 4. Convert Single File

Convert a single MP4 file to MP3. Update the `input_file` variable with your filename.

In [ ]:
# Configure your input file here
input_file = "your_video.mp4"  # Change this to your uploaded filename

print("=" * 60)
print("Single File Conversion")
print("=" * 60)
print()

if os.path.exists(input_file):
    result = convert_mp4_to_mp3(input_file)
    if result:
        print()
        print("=" * 60)
        print("✓ Conversion successful!")
        print("=" * 60)
else:
    print(f"✗ Error: File '{input_file}' not found")
    print("\nAvailable files:")
    for f in os.listdir('.'):
        if f.endswith('.mp4'):
            print(f"  - {f}")

## 5. Convert All Uploaded Files (Batch)

Convert all MP4 files in the current directory to MP3.

In [ ]:
print("=" * 60)
print("Batch Conversion")
print("=" * 60)
print()

# Convert all MP4 files in current directory
success, fail, skip = batch_convert("*.mp4")

print("=" * 60)
print("Batch Conversion Completed!")
print("=" * 60)
print(f"✓ Success: {success} file(s)")
print(f"✗ Failed: {fail} file(s)")
print(f"⊘ Skipped: {skip} file(s)")
print("=" * 60)

## 6. Download Converted MP3 Files

Download all converted MP3 files to your computer.

In [ ]:
from google.colab import files
import glob

print("=" * 60)
print("Download MP3 Files")
print("=" * 60)
print()

# Find all MP3 files
mp3_files = glob.glob("*.mp3")

if mp3_files:
    print(f"Found {len(mp3_files)} MP3 file(s)")
    print()
    
    for mp3_file in mp3_files:
        print(f"Downloading: {mp3_file}")
        files.download(mp3_file)
    
    print()
    print("=" * 60)
    print("✓ All files downloaded!")
    print("=" * 60)
else:
    print("✗ No MP3 files found")
    print("Please run the conversion cells first.")

## 7. Cleanup (Optional)

Remove all uploaded and converted files from Colab.

In [ ]:
import os
import glob

print("=" * 60)
print("Cleanup")
print("=" * 60)
print()

# Remove all MP4 and MP3 files
files_to_remove = glob.glob("*.mp4") + glob.glob("*.mp3")

if files_to_remove:
    for file in files_to_remove:
        os.remove(file)
        print(f"✓ Removed: {file}")
    
    print()
    print("=" * 60)
    print(f"✓ Cleanup completed! Removed {len(files_to_remove)} file(s)")
    print("=" * 60)
else:
    print("No files to clean up.")